# Step 1: Install Dependencies

In [1]:
# Install necessary libraries for text classification and deployment
!pip install -q transformers datasets gradio

# Step 2: Load & Prepare Dataset

In [2]:
from datasets import Dataset

# Define AI-generated text samples (from ChatGPT, DeepSeek, Claude, etc.)
ai_text_samples = [
    "Artificial intelligence is revolutionizing industries by automating processes.",
    "The universe is vast, with countless galaxies and undiscovered planets.",
    "Quantum computing has the potential to solve complex problems exponentially faster.",
    "AI-generated text can sometimes mimic human writing patterns convincingly."
]

# Define human-written text samples (from Wikipedia, books, articles)
human_text_samples = [
    "The history of ancient civilizations dates back thousands of years.",
    "Cooking requires a balance of flavors, textures, and techniques.",
    "A compelling novel keeps readers engaged with strong characters and plot development.",
    "Traveling allows individuals to experience different cultures and traditions."
]

# Create labels for classification (1 = AI-generated, 0 = Human-written)
labels = [1] * len(ai_text_samples) + [0] * len(human_text_samples)

# Convert data into a Hugging Face Dataset format
dataset = Dataset.from_dict({"text": ai_text_samples + human_text_samples, "label": labels})

# Split dataset into training (80%) and testing (20%) sets
dataset = dataset.train_test_split(test_size=0.2)

# Step 3: Load & Tokenize `ModernBERT-base`

In [4]:
from transformers import AutoTokenizer

# ModernBERT model
model_name = "answerdotai/ModernBERT-base"

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize dataset with padding and truncation
tokenized_datasets = dataset.map(lambda x: tokenizer(x["text"],
                                                     padding="max_length",
                                                     truncation=True,
                                                     max_length=128), batched=True)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

# Step 4: Fine-Tune `ModernBERT-base`

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load ModernBERT model for binary classification (2 labels: AI vs. Human)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define training parameters
training_args = TrainingArguments(
    output_dir="./ai_detector",  # Directory to save model checkpoints
    learning_rate=0.001,  # Optimal learning rate for fine-tuning
    weight_decay=0.01,  # Regularization to prevent overfitting
    per_device_train_batch_size=2,  # Batch size for training
    per_device_eval_batch_size=2,  # Batch size for evaluation
    num_train_epochs=3,  # Number of training iterations over dataset
    eval_strategy="epoch",  # Evaluate model at the end of each epoch
    save_strategy="epoch",  # Save model at the end of each epoch
    logging_strategy="epoch",  # Log training/evaluation loss at each epoch
    logging_steps=1,  # Log training loss every step (for debugging)
    logging_dir="./logs",  # Directory to store training logs
    load_best_model_at_end=True,  # Load best model checkpoint after training
    report_to=["wandb"],  # Log training results to Weights & Biases
    # push_to_hub=True,  # Uncomment to upload model to Hugging Face Hub
)

# Initialize Trainer for model fine-tuning
trainer = Trainer(
    model=model,  # ModernBERT model
    args=training_args,  # Training configuration
    train_dataset=tokenized_datasets["train"],  # Training data
    eval_dataset=tokenized_datasets["test"],  # Evaluation data
    processing_class=tokenizer  # Tokenizer for processing text
)

# Start training the model
trainer.train()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: antararane20 (antararane20-shri-bhagubhai-mafatlal-polytechnic) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W1013 17:59:19.926000 222 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss
1,9.474800,2.754298
2,1.905700,2.028544
3,1.408800,0.789896


TrainOutput(global_step=9, training_loss=4.263119803534614, metrics={'train_runtime': 213.9227, 'train_samples_per_second': 0.084, 'train_steps_per_second': 0.042, 'total_flos': 1533410307072.0, 'train_loss': 4.263119803534614, 'epoch': 3.0})

# Step 5: Test the Model

In [6]:
import numpy as np
from transformers import pipeline

# Load trained model as a text classifier pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Example AI-generated text for testing
test_text = "Deep learning is a subset of machine learning that uses artificial neural networks to model and solve complex problems. It is characterized by deep architectures with multiple layers of interconnected neurons, allowing it to automatically learn hierarchical representations from data."

# Run text classification
result = classifier(test_text)

# Convert output label to human-readable format
label = "AI-Generated" if result[0]["label"] == "LABEL_1" else "Human-Written"
confidence = np.round(result[0]["score"], 3)

# Print classification result
print(f"Prediction: {label} (Confidence: {confidence})")

Device set to use cuda:0


Prediction: Human-Written (Confidence: 0.7)


# Step 6: Deploy as a Gradio Web App

In [7]:
import gradio as gr

# Define function for real-time AI text detection
def predict_ai_text(input_text):
    result = classifier(input_text)
    label = "AI-Generated" if result[0]["label"] == "LABEL_1" else "Human-Written"
    confidence = np.round(result[0]["score"], 3)
    return f"{label} (Confidence: {confidence})"

# Create Gradio interface for web-based text detection
gr.Interface(fn=predict_ai_text, inputs="text", outputs="text", title="AI Text Detector").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b9614fcacb8c164be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
